# SLT Notebook 6 — Learning zero at all widths

**Singular Learning Theory, Iliad Intensive 2026**

Worked solutions, and the output of every code cell, are on the reading page [`slt6_learning_zero.html`](slt6_learning_zero.html) (open it in a web browser).

*Question:* what is the learning coefficient of $H$ cosine neurons learning the zero function, and how does it depend on $H$?

This is the challenge problem set for the second effective-theory lecture. It uses two things from Notebook 5: the Hermite coordinates of a cosine neuron (Exercise 5.2a) and Newton polygons (Exercise 5.3a). Notebook 5's optional §5.4 (tropical arithmetic) is useful background for 6.1d–f.

**Notation.** As in Notebook 5: inputs $x\sim\mathcal N(0,1)$, the cosine network $f_\theta(x)=\sum_{i=1}^Ha_i\cos(w_ix)$ with $\theta=(a,w)\in[-1,1]^{2H}$ and the uniform prior, $K(\theta)=\|f_\theta\|^2$ (learning zero), $Z_n=\int\varphi\,e^{-nK}$, $F_n=-\log Z_n$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

<a id="slt6-s1"></a>

## 6.1 Learning zero with $H$ neurons

*Challenge section.* Difficulty ●●●◐○ · Importance ●●●●○

*Question:* what is the learning coefficient $\lambda_H$ of learning zero with $H$ cosine neurons?

*Notation:* By Notebook 5, Exercise 5.2a.1, learning zero is $K=\sum_{j\ge0}M_j^2/(2j)!$ with moments $M_j=\sum_i\tilde a_iw_i^{2j}$, $\tilde a_i=a_ie^{-w_i^2/2}$. Throughout this section the parameters are $(\tilde a,w)$ with the uniform prior on $[-1,1]^{2H}$, and we drop the tildes. (The cosine network with its own prior differs by the factor $e^{-w^2/2}\in[e^{-1/2},1]$ and a slightly different box; this does not affect anything below.) $u_i=w_i^2$. $N$ is the number of moments kept, $L=\tfrac12\log n$.

The answer to be established:
$$\lambda_H=\min\Big(\frac H2,\ \min_{1\le k\le H-1}\Big[\frac{k+1}4+\frac H{4k}\Big]\Big)=\tfrac12\sqrt H+O(1).$$
The first term matters only for $H\le2$. The upper bound is a construction (Exercise 6.1b). The lower bound goes through a Fourier dual in which the $H$ neurons enter through an $H$-th power of a single-neuron function (Exercise 6.1c); a tropical estimate of that function (6.1d) turns the problem into a linear program (6.1e). One step, controlling cancellations inside a single neuron, is left as a starred exercise (6.1f). Exercise 6.1g is the punchline: truncated to $k_0+1\lesssim\sqrt H$ field coordinates, the model has $\lambda=\tfrac12(k_0+1)$, exactly as if it were regular in field space.

**Exercise 6.1a (Finitely many moments suffice).**

1. Let $e_l$ be the elementary symmetric polynomials of $u_1,\dots,u_H$. Show that for all $j\ge0$
$$M_{j+H}=\sum_{l=1}^H(-1)^{l+1}e_l\,M_{j+H-l}.$$
2. Deduce $|M_j|\le(H2^H)^{j-H+1}\max_{l<H}|M_l|$ for $j\ge H$.
3. Conclude $c\,K_{\rm mom}\le K\le C\,K_{\rm mom}$ on the whole box, with $K_{\rm mom}=\sum_{j=0}^{H-1}M_j^2$ and constants depending only on $H$. Hence $\lambda_H$ is the learning coefficient of $K_{\rm mom}$, a polynomial in which every $M_j$ has degree at most $2H-1$.

<details><summary>Hint</summary>

Part 1: each $u_i$ is a root of $\prod_{i'}(u-u_{i'})=\sum_{l=0}^H(-1)^le_lu^{H-l}$; multiply by $a_iu_i^j$ and sum over $i$. Part 3: $\sum_m x^{2m}/(2m)!=\cosh x$.
</details>

**Exercise 6.1b (Upper bound by construction).**

1. Show $\lambda_H\le H/2$ (Notebook 5, Exercise 5.2d, with $N=H$).
2. Fix $1\le k\le H-1$ and $s=n^{-1/(4k)}$. Show that if $|w_i|\le s$ for all $i$, then $|M_j|\le Hn^{-1/2}$ for all $j\ge k$.
3. For $j<k$ write $M_j=s^{2j}m_j$ with $m_j=\sum_ia_iv_i^j$, $v_i=u_i/s^2\in[0,1]$. Show that it suffices to impose $|m_j|\le n^{-\frac12(1-j/k)}$ for $j<k$ to get $nK_{\rm mom}=O(1)$.
4. Restrict $v_1,\dots,v_k$ to fixed disjoint subintervals of $[0,1]$ and $|a_i|\le\delta$ for $i>k$, with $\delta$ small depending on $H,k$. Show that for each such choice of $w$ and of $a_{k+1},\dots,a_H$, the set of $(a_1,\dots,a_k)\in[-1,1]^k$ satisfying the constraints of part 3 has volume $\ge c\prod_{j<k}n^{-\frac12(1-j/k)}=c\,n^{-(k+1)/4}$.
5. Conclude $V(C/n)\ge c\,n^{-H/(4k)-(k+1)/4}$, hence $\lambda_H\le\frac{k+1}4+\frac H{4k}$.
6. A simpler construction imposes $|M_j|\le n^{-1/2}$ on $a$ directly for $j<k$, without using the smallness of $w$. What bound does it give, and why is it worse?

<details><summary>Hint</summary>

Part 4: for fixed $v$, $(a_1,\dots,a_k)\mapsto(m_j)_{j<k}$ is the Vandermonde matrix $V_k$ of $v_1,\dots,v_k$ applied to $(a_1,\dots,a_k)$, plus a shift by the contribution of the other neurons. On the restricted set, $|\det V_k|$ is bounded below, so $V_k^{-1}$ is bounded; choose $\delta$ so that the shift, pulled back by $V_k^{-1}$, stays well inside $[-1,1]^k$.
</details>

**Exercise 6.1c (The Fourier dual).** Let $N=H$, $\phi(a,w)=a\,(1,w^2,\dots,w^{2(N-1)})\in\mathbb R^N$, so $K_{\rm mom}(\theta)=\big|\sum_{i=1}^H\phi(a_i,w_i)\big|^2$.

1. Show, for $y\in\mathbb R^N$,
$$e^{-n|y|^2}=(4\pi n)^{-N/2}\int_{\mathbb R^N}e^{i\langle\eta,y\rangle-|\eta|^2/(4n)}\,d\eta .$$
2. Deduce
$$\bar Z_n=(4\pi n)^{-N/2}\int_{\mathbb R^N}\chi(\eta)^H\,e^{-|\eta|^2/(4n)}\,d\eta,\qquad\chi(\eta)=\mathbb E_{(a,w)}\,e^{i\langle\eta,\phi(a,w)\rangle},$$
the expectation over one neuron with the uniform prior.
3. Show $\chi(\eta)=\mathbb E_w\,\mathrm{sinc}\big(P_\eta(w^2)\big)$, where $P_\eta(u)=\sum_{j<N}\eta_ju^j$ and $\mathrm{sinc}(z)=\sin z/z$. Conclude that $\chi$ is real, $|\chi|\le1$, and
$$\bar Z_n\le(4\pi n)^{-N/2}\int|\chi(\eta)|^He^{-|\eta|^2/(4n)}d\eta,\qquad |\chi(\eta)|\le\mathbb E_w\min\big(1,|P_\eta(w^2)|^{-1}\big).$$
4. Check $H=1$: compute $\bar Z_n$ exactly from part 2 and recover $\lambda_1=\tfrac12$.
5. Explain in one sentence what the dual has done to the cancellations between neurons noted at the end of Notebook 5, §5.3.

<details><summary>Hint</summary>

Part 1: complete the square, or recall that the Fourier transform of a Gaussian is a Gaussian. Part 3: integrate over $a\in[-1,1]$ first; $\langle\eta,\phi(a,w)\rangle=a\,P_\eta(w^2)$. Part 4: $\int_{\mathbb R}\mathrm{sinc}=\pi$.
</details>

**Exercise 6.1d (One neuron, tropically).** Write $\ell_j=\log_+|\eta_j|=\max(0,\log|\eta_j|)$ and
$$\Phi(\ell)=\min_{y\le0}\Big[\max\big(0,\max_j(\ell_j+2jy)\big)-y\Big].$$

1. Suppose $|P_\eta(u)|\ge c\,\max_j|\eta_j|u^j$ on $[0,1]$ for some $c\in(0,1]$ (no cancellation). Show $|\chi(\eta)|\le\frac2c\,e^{-\Phi(\ell)}$.
2. (Staircase.) For $0<Y\le L/2$ let $\ell_0=0$ and $\ell_j=\min(L,2jY)$ for $j\ge1$. Show $\Phi(\ell)=Y$. What do these $\eta$ correspond to, as polynomials $P_\eta$?
3. Why is $|\chi(\eta)|$ not approximated by the measure of $\{w:|P_\eta(w^2)|<1\}$? Compare the two for $P_\eta(u)=r^{1/2}+ru$.

<details><summary>Hint</summary>

Part 1: substitute $w=e^y$, so $dw=e^ydy$, and note that $y\mapsto\max(0,\max_j(\ell_j+2jy))-y$ is convex and piecewise linear with slopes in $\{-1,1,3,5,\dots\}$, never $0$. Coordinates with $|\eta_j|<1$ only contribute terms that are already below the $0$ in the max.
</details>

**Exercise 6.1e (The linear program).**

1. Split $\mathbb R^N$ into the regions $\{|\eta_j|\in[e^{\ell_j},e^{\ell_j+1})\}$ (with $|\eta_j|<1$ for $\ell_j=0$). Assuming the bound of 6.1d.1 for every $\eta$, show
$$\log\bar Z_n\le-NL+\max_{\ell\in[0,L]^N}\Big[\sum_j\ell_j-H\,\Phi(\ell)\Big]+O(\log\log n),$$
so that $\lambda_H\ge\frac1{2L}\big(NL-\max[\dots]\big)$ asymptotically.
2. Show that $\Phi$ is convex, so the maximisation is a linear program in $(\ell,y,t)$: maximise $\sum_j\ell_j-Ht$ subject to $t\ge-y$, $t\ge\ell_j+2jy-y$, $y\le0$, $0\le\ell_j\le L$.
3. Show that the optimum is attained on a staircase of Exercise 6.1d.2 with $Y=L/(2k)$ for some $k\in\{1,\dots,N-1\}$, or at $\ell=0$. Conclude
$$\lambda_H\ge\min\Big(\frac N2,\ \min_{1\le k\le N-1}\Big[\frac{k+1}4+\frac H{4k}\Big]\Big),$$
which with $N=H$ matches Exercise 6.1b.
4. Show $\lambda_H=\tfrac12\sqrt H+O(1)$ and describe the dominant stratum.

<details><summary>Hint</summary>

Part 1: a region with given $\ell$ has volume $\asymp e^{\sum\ell_j}$; the Gaussian factor kills $\ell_j\ge L+\log L$; there are $O((\log n)^N)$ regions. Part 3: $\Phi(\ell)\le Y$ iff there is $y'\in[0,Y]$ with $\ell_j\le2jy'+Y-y'$ for all $j$. For fixed $Y$, maximise $\sum_j\min(L,2jy'+Y-y')$ over $y'$; then the objective is a concave piecewise linear function of $Y$ with breakpoints at $Y=L/(2k)$.
</details>

**The one-neuron picture.** Each stratum of dual vectors is a point $(q,y)$: $q$ is the *valuation* of the one-neuron function on the stratum, $|\chi|\approx r^{-q}$ at $|\eta|\approx r$, and $y\le0$ is the stratum's log-volume relative to the ball of radius $r$, per unit of $\log r$. For $\deg P_\eta\le4$ ($N=5$) the points are the staircases of Exercise 6.1d.2, where $P_\eta$ vanishes to order $k$ at $u=0$, at $\big(\tfrac1{2k},-\tfrac{k+1}2\big)$ for $k=1,\dots,4$, and the bounded region $|\eta|=O(1)$ at $(0,-N)$. (Generic $\eta$ sit at $(1,0)$, off the left panel; they matter only for $H\le2$.)

The width enters only through $|\chi|^H$, which shears the picture, $(q,y)\mapsto(q,\,y-Hq)$; by Exercise 6.1e the highest point after the shear gives $\lambda_H=-\tfrac12\max(y-Hq)$. Unsheared, this is the point where a line of slope $H$ touches the configuration from above (left panel). As $H$ grows the touching point moves down the staircase, switching from $k$ to $k+1$ at $H=k(k+1)$, and at $H=N(N-1)=20$ it reaches the bounded region, after which $\lambda=N/2$: Exercise 6.1g. Dually, each stratum is a line $\lambda=\tfrac12(Hq+|y|)$ in the width, and $\lambda_H$ is their lower envelope (right panel). The cell computes the valuations numerically, from $\mathbb E_w\min(1,|P_\eta(w^2)|^{-1})$ at two scales $r$; the volumes are the constraint counts of Exercise 6.1d.2.

In [ ]:
N = 5                                    # P_eta has degree <= 4
def chi_bound(eta):
    # E_w min(1, 1/|P_eta(w^2)|), w uniform on [0,1], integrated on a log grid in w
    lw = np.linspace(-40, 0, 40001); w = np.exp(lw)
    P = sum(e * w**(2 * j) for j, e in enumerate(eta))
    return np.sum(np.minimum(1, 1 / np.abs(P)) * w) * (lw[1] - lw[0])
def staircase(r, k):
    return [1.0] + [r**min(1.0, j / k) for j in range(1, N)]
r1, r2 = 1e8, 1e16
q_num = np.array([-np.log(chi_bound(staircase(r2, k)) / chi_bound(staircase(r1, k))) / np.log(r2 / r1) for k in range(1, N)])
print("valuations, k=1..4:", np.round(q_num, 4), "  vs 1/(2k):", np.round(1 / (2 * np.arange(1, N)), 4))

# strata: vanishing order k at u=0 (k=1..N-1), and the bounded region |eta|=O(1)
labels = [f"$k={k}$" for k in range(1, N)] + ["bounded"]
q = np.r_[q_num, 0.0]
y = np.r_[-(np.arange(1, N) + 1) / 2, -N]
Hs, cols = [4, 9, 16, 30], ["C0", "C1", "C2", "C3"]

fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4.3))
ax.plot(q, y, "ko", ms=5, zorder=3)
for lab, qq, yy in zip(labels, q, y):
    ax.annotate(lab, (qq, yy), textcoords="offset points", xytext=(7, -3), fontsize=9)
qs = np.array([-0.02, 0.56])
for H, c in zip(Hs, cols):
    i = np.argmax(y - H * q)                          # highest point after the shear (q,y) -> (q, y - H q)
    ax.plot(qs, y[i] + H * (qs - q[i]), c=c, lw=1.2, label=fr"$H={H}$: touches {labels[i]}, $\lambda_H={-(y[i]-H*q[i])/2:.2f}$")
    ax.plot(q[i], y[i], "o", mfc="none", mec=c, ms=13, mew=1.5)
ax.set_xlim(-0.02, 0.56); ax.set_ylim(-N - 0.6, 0.3)
ax.set_xlabel(r"valuation $q$  ($|\chi|\approx r^{-q}$)"); ax.set_ylabel(r"log-volume per $\log r$")
ax.set_title(r"one neuron, $\deg P\leq 4$: lines of slope $H$ from above"); ax.legend(fontsize=8, loc="lower right")

H = np.linspace(1, 36, 400)
lines = [(k + 1) / 4 + H / (4 * k) for k in range(1, N)] + [np.full_like(H, N / 2), H / 2]   # last: generic eta, the point (1,0)
for k, (L, lab) in enumerate(zip(lines, labels + ["generic"])):
    bx.plot(H, L, lw=1, alpha=0.6, label=lab)
bx.plot(H, np.min(lines, axis=0), "k", lw=2.2, label=r"$\lambda_H$ = lower envelope")
for kk in range(1, N): bx.axvline(kk * (kk + 1), c="gray", ls=":", lw=0.8)
bx.set_ylim(0, N / 2 + 1); bx.set_xlabel("width $H$"); bx.set_ylabel(r"$\lambda$")
bx.set_title(r"each stratum is a line in $H$; switches at $H=k(k+1)$"); bx.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

**Exercise 6.1f\* (The missing step: cancellation inside one neuron).** The assumption of 6.1d.1 fails when $P_\eta$ has roots of high order in $(0,1]$.

1. (Sublevel sets.) Let $g\in C^m(I)$ with $|g^{(m)}|\ge\mu>0$ on an interval $I$. Prove $|\{x\in I:|g(x)|<\delta\}|\le C_m(\delta/\mu)^{1/m}$.
2. (Heuristic.) Suppose $P_\eta$ has a root of order $k$ at $u_0\in(0,1]$, at scale $|\eta|\approx r$. Argue that $|\chi(\eta)|\approx r^{-1/k}$, rather than $r^{-1/(2k)}$ for a root at $u_0=0$, and that such $\eta$ form a set of relative log-volume about $\big(\tfrac{k+1}2-\tfrac1k\big)\log r$. Show that the resulting candidate value $\frac H{2k}+\frac{k+1}4-\frac1{2k}$ is never below the $u_0=0$ value $\frac H{4k}+\frac{k+1}4$ when $H\ge2$.
3. (Open-ended.) Turn 1 and 2 into a proof of the lower bound in Exercise 6.1e.

<details><summary>Hint</summary>

Part 1: induction on $m$. The set where $|g^{(m-1)}|<\beta$ is an interval of length $\le2\beta/\mu$; on each of the at most two remaining intervals, apply the case $m-1$ with $\mu$ replaced by $\beta$; then choose $\beta=\mu^{(m-1)/m}\delta^{1/m}$. Part 2: near $w_0=\sqrt{u_0}\ne0$ the map $w\mapsto w^2$ is a diffeomorphism, so $w$-measure equals $u$-measure up to constants; the position $u_0$ is a free parameter, which returns one factor of $r^{1/k}$ in volume.
</details>

**Exercise 6.1g (A truncated field space sees only its dimension).** Truncate learning zero to $\mathcal F_{\le2k_0}$: keep $N=k_0+1\le H$ moments $M_0,\dots,M_{k_0}$.

1. Show $\lambda_{\le2k_0}\le\frac{k_0+1}2$.
2. Running Exercises 6.1c–6.1e with $N=k_0+1$ gives $\lambda_{\le2k_0}\ge\min\big(\frac{k_0+1}2,\ \min_{1\le k\le k_0}\big[\frac{k+1}4+\frac H{4k}\big]\big)$. Show this equals $\frac{k_0+1}2$ if and only if $H\ge k_0(k_0+1)$. So for $k_0\lesssim\sqrt H$, the truncated model has learning coefficient half its field-space dimension, although it is highly singular in parameter space.
3. The boundary case $H=k_0(k_0+1)$ with $H=2$, $k_0=1$: compute directly that $K_{\rm mom}=(a_1+a_2)^2+(a_1w_1^2+a_2w_2^2)^2$ has $\lambda=1$ and $m=3$.
4. (Heuristic synthesis with Notebook 5, §5.2.) At sample size $n$ only about $k_0(n)\approx\log n/(2\log\log n)$ moments are resolved (Notebook 5, Exercise 5.2a). Argue that $\lambda_{\rm eff}(n)$ grows roughly like $\tfrac12(k_0(n)+1)$ while $k_0(n)(k_0(n)+1)\le H$, and saturates near $\tfrac12\sqrt H$ once $\log n\approx\sqrt H\log H$. Why is this not a theorem as stated?

<details><summary>Hint</summary>

Part 2: $k\mapsto\frac{k+1}4+\frac H{4k}$ is convex and decreasing for $k<\sqrt H$. Part 3: with $s=a_1+a_2$, $a_1w_1^2+a_2w_2^2=a_1(w_1^2-w_2^2)+sw_2^2$; show $K_{\rm mom}\asymp s^2+a_1^2(w_1-w_2)^2(w_1+w_2)^2$ and note that $\{s^2<t/2\}\times\{g^2<t/2\}\subset\{s^2+g^2<t\}\subset\{s^2<t\}\times\{g^2<t\}$ for $g=a_1(w_1-w_2)(w_1+w_2)$; then use the monomial rule of Notebook 2 Exercise 2.2.2.
</details>

**Code: the linear program, the staircase bound, and truncation.** The first cell solves the LP of Exercise 6.1e numerically and compares with the closed form; the second checks the staircase estimate $|\chi|\approx r^{-1/(2k)}$ of Exercise 6.1d by direct integration; the third tabulates Exercise 6.1g.

In [ ]:
def lam_lp(N, H, L=1.0):
    # maximise sum(l) - H t  s.t.  t >= -y,  t >= l_j + (2j-1) y,  y <= 0,  0 <= l_j <= L ; return (N L - max)/(2L)
    nv = N + 2; c = np.zeros(nv); c[:N] = -1; c[N + 1] = H
    A = [np.r_[np.zeros(N), -1, -1]] + [np.r_[np.eye(N)[j], 2 * j - 1, -1] for j in range(N)]
    res = linprog(c, A_ub=np.array(A), b_ub=np.zeros(N + 1), bounds=[(0, L)] * N + [(None, 0), (None, None)])
    return (N * L + res.fun) / (2 * L)

def lam_closed(N, H):
    return min([N / 2] + [(k + 1) / 4 + H / (4 * k) for k in range(1, N)])

Hs = np.arange(1, 61)
lp = [lam_lp(H, H) for H in Hs]
print("max |LP - closed form| over H<=60:", max(abs(l - lam_closed(H, H)) for l, H in zip(lp, Hs)))
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(Hs, lp, "o", ms=3, label=r"LP value of $\lambda_H$")
ax.plot(Hs, 0.5 * np.sqrt(Hs) + 0.25, label=r"$\frac{1}{2}\sqrt{H}+\frac{1}{4}$")
ax.set_xlabel("$H$"); ax.legend(); plt.show()

In [ ]:
def chi(eta, m=2_000_001):
    # E_w sinc(P_eta(w^2)), w uniform on [-1,1]; log-spaced nodes near 0 plus a uniform grid
    w = np.concatenate([np.logspace(-12, -3, 200001), np.linspace(1e-3, 1, m)])
    P = sum(e * w**(2 * j) for j, e in enumerate(eta))
    trap = getattr(np, "trapezoid", None) or np.trapz
    return trap(np.sinc(P / np.pi), w)
k = 2
for r in [1e4, 1e6, 1e8, 1e10]:
    eta = [1.0] + [r**(j / k) for j in range(1, k + 1)]      # staircase with Y = L/(2k)
    print(f"r=1e{int(np.log10(r)):>2}:  |chi| = {abs(chi(eta)):.3e}    r^(-1/(2k)) = {r**(-1/(2*k)):.3e}")

In [ ]:
for H in [6, 12, 20, 30]:
    row = []
    for k0 in range(1, 7):
        N = k0 + 1
        if N > H: break
        row.append(f"k0={k0}: {lam_lp(N, H):.3f}" + ("" if abs(lam_lp(N, H) - N / 2) < 1e-9 else " (< N/2)"))
    print(f"H={H:2d} | " + " | ".join(row))